# First prototype for HyperBubble Resolution-Oriented GNN

In [19]:
!pip install torch_geometric

from pathlib import Path
from typing import Any, Dict, List, Optional
import json
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: C:\Users\Przemek\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [20]:
NUC2ID = { 'A':1, 'C':2, 'G':3, 'T':4, 'N':5 }
def seq_to_tokens(seq: str) -> torch.LongTensor:
    return torch.tensor([NUC2ID.get(c, 5) for c in seq], dtype=torch.long)

def _safe_get(d: Dict[str, Any], key: str, default):
    v = d.get(key, default)
    return default if v is None else v

# reverse mapping for debug
ID2NUC = {v:k for k,v in NUC2ID.items()}

def tokens_to_seq(tokens: torch.LongTensor) -> str:
    # tokens is 1D LongTensor, e.g. [21] (a single sequence)
    return "".join(ID2NUC.get(int(t), "N") for t in tokens.tolist())

In [21]:
class HyperbubbleDataset(Dataset):
    """
    Minimal GNN-ready dataset from your JSONL:
      - Node features:
          seq_tokens : [N, K] Long (embed this; no one-hot)
          x_cov      : [N, 1] Float (KC coverage; 0 if unknown)
      - Graph:
          edge_index : [2, E] Long
          edge_attr  : [E, 5] Float (len_nodes, len_bp, cov_min, cov_mean, on_ref)
          start_idx, end_idx : Long
      - Labels:
          y_edge_mask : [E] Long (1 on labeled path edges, else 0)
          label_path_idx : Long (-1 if none)
    """
    def __init__(self, files: List[str]):
        self.files = [Path(p) for p in files]
        self.records: List[Dict[str, Any]] = []
        for fp in self.files:
            with fp.open("r") as fh:
                for line in fh:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        self.records.append(json.loads(line))
                    except json.JSONDecodeError:
                        continue

    def __len__(self) -> int:
        return len(self.records)

    def _build_graph(self, rec: Dict[str, Any]) -> Data:
        # --- nodes (map seq -> idx), carry best-known coverage ---
        node_seqs: Dict[str, int] = {}
        node_covs: List[float] = []

        def ensure_node(seq: str, cov: Optional[float] = None) -> int:
            if seq not in node_seqs:
                node_seqs[seq] = len(node_seqs)
                node_covs.append(float(cov) if cov is not None else 0.0)
            else:
                i = node_seqs[seq]
                if cov is not None and node_covs[i] == 0.0:
                    node_covs[i] = float(cov)
            return node_seqs[seq]

        # endpoints
        start_seq = rec["start_seq"]
        end_seq   = rec["end_seq"]

        # nodes with cov
        for n in _safe_get(rec, "nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))

        # include 1-hop context nodes (optional cov)
        for n in _safe_get(rec, "upstream_nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))
        for n in _safe_get(rec, "downstream_nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))

        # make sure endpoints are present
        ensure_node(start_seq)
        ensure_node(end_seq)

        # --- edges + attributes ---
        edge_src, edge_dst, edge_attr = [], [], []
        for e in _safe_get(rec, "edges", []):
            u = ensure_node(e["source_seq"])
            v = ensure_node(e["target_seq"])
            edge_src.append(u)
            edge_dst.append(v)
            edge_attr.append([
                float(e.get("len_nodes", 0)),
                float(e.get("len_bp", 0)),
                float(e.get("cov_min", 0)),
                float(e.get("cov_mean", 0.0)),
                1.0 if e.get("on_ref", False) else 0.0
            ])

        # --- node features ---
        node_order = [None] * len(node_seqs)
        for s, idx in node_seqs.items():
            node_order[idx] = s

        # k is constant across the dataset -> sequences are k-mers
        seq_tokens = torch.stack([seq_to_tokens(s) for s in node_order], dim=0)  # [N, K]
        x_cov = torch.tensor(node_covs, dtype=torch.float32).unsqueeze(1)        # [N, 1]

        start_idx = torch.tensor(node_seqs[start_seq], dtype=torch.long)
        end_idx   = torch.tensor(node_seqs[end_seq], dtype=torch.long)

        edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
        edge_attr  = (torch.tensor(edge_attr, dtype=torch.float32)
                      if edge_attr else torch.zeros((0,5), dtype=torch.float32))

        # --- labels from label_path ---
        num_edges = edge_index.size(1)
        y_edge_mask = torch.zeros(num_edges, dtype=torch.long)
        label_path_idx = -1
        paths_list = _safe_get(rec, "paths", [])
        lp = rec.get("label_path")  # may be None
        if lp is not None and 0 <= lp < len(paths_list) and num_edges > 0:
            label_path_idx = int(lp)
            y_edge_mask[torch.tensor(paths_list[label_path_idx], dtype=torch.long)] = 1

        data = Data(
            # node features
            seq_tokens=seq_tokens,    # Long [N, K]
            x_cov=x_cov,              # Float [N, 1]
            # graph
            edge_index=edge_index,    # Long [2, E]
            edge_attr=edge_attr,      # Float [E, 5]
            start_idx=start_idx,      # Long []
            end_idx=end_idx,          # Long []
            num_nodes=seq_tokens.size(0),
            # labels
            y_edge_mask=y_edge_mask,                      # Long [E]
            label_path_idx=torch.tensor(label_path_idx)   # Long []
        )
        # keep only tensors in Data to avoid collation issues
        # (bubble_id/k useful but optional)
        if "bubble_id" in rec:
            data.bubble_id = torch.tensor(rec["bubble_id"], dtype=torch.long)
        if "k" in rec:
            data.k = torch.tensor(rec["k"], dtype=torch.long)
        return data

    def __getitem__(self, idx: int) -> Data:
        return self._build_graph(self.records[idx])


In [22]:
# Point to one or more JSONL files from your pipeline
jsonl_paths = [
    "./sample_data.jsonl",
]

dataset = HyperbubbleDataset(jsonl_paths)

# Peek
sample = dataset[0]
print("num_nodes:", sample.num_nodes)
print("num_edges:", sample.edge_index.size(1))

for i, (tok_row, cov) in enumerate(zip(sample.seq_tokens, sample.x_cov)):
    seq = tokens_to_seq(tok_row)
    print(f"node {i}: seq={seq}, cov={cov.item()}")

# Dataloader for batching graphs
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0)

num_nodes: 6
num_edges: 4
node 0: seq=CGGGCGGACCATCGACTGGCT, cov=24.0
node 1: seq=CGTACCGGGCGGACCATCGAC, cov=24.0
node 2: seq=GCGGTGGGCTACCTGAATCAC, cov=25.0
node 3: seq=ATCGGGCGGTGGGCTACCTGA, cov=14.0
node 4: seq=GGCGGTGGGCTACCTGAATCA, cov=21.0
node 5: seq=GTACCGGGCGGACCATCGACT, cov=25.0


In [23]:
# --- device selection (CPU / CUDA / DirectML) ---
USE_DIRECTML = True  # set False to stay on CPU; CUDA will be auto-used if available and USE_DIRECTML is False

import torch
device = torch.device('cpu')
if not USE_DIRECTML and torch.cuda.is_available():
    device = torch.device('cuda')
elif USE_DIRECTML:
    try:
        import torch_directml
        device = torch_directml.device()
        print("Using DirectML:", device)
    except Exception as e:
        print("DirectML not available, falling back to CPU:", e)
        device = torch.device('cpu')

# --- simple embedding + dense GCN that avoids PyG custom ops ---
import torch.nn as nn
import torch.nn.functional as F

class DenseGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)

    def forward(self, H, edge_index_local, n_nodes: int):
        if n_nodes == 0:
            return H
        A = H.new_zeros((n_nodes, n_nodes))
        if edge_index_local.numel() > 0:
            src, dst = edge_index_local
            one = torch.ones_like(src, dtype=H.dtype)
            A.index_put_((src, dst), one, accumulate=True)

        # add self-loops w/o torch.eye
        idx = torch.arange(n_nodes, device=H.device)
        try:
            A[idx, idx] += 1
        except Exception:
            flat = A.view(-1)
            diag_idx = torch.arange(0, n_nodes*n_nodes, step=n_nodes+1, device=H.device)
            flat.index_add_(0, diag_idx, torch.ones(n_nodes, device=H.device, dtype=H.dtype))
            A = flat.view(n_nodes, n_nodes)

        deg = A.sum(dim=1) + 1e-8
        D_inv_sqrt = deg.pow(-0.5)
        A_norm = (D_inv_sqrt[:, None] * A) * D_inv_sqrt[None, :]
        return A_norm @ self.lin(H)

# ---- Simple GNN with sequence embedding + GCN + edge MLP ----
class HyperbubbleGNN(nn.Module):
    def __init__(
        self,
        vocab_size=5,          # tokens: 0=PAD, A,C,G,T (and optionally N)
        embed_dim=32,          # per-token embedding dim
        gcn_hidden=64,         # node hidden dim (consistent throughout)
        edge_hidden=64,        # hidden in edge MLP
        edge_feat_dim=5,       # [len_nodes, len_bp, cov_min, cov_mean, on_ref]
        use_lstm=False,        # optional: token encoder = mean or BiLSTM
    ):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.use_lstm = use_lstm
        if use_lstm:
            self.lstm = nn.LSTM(embed_dim, gcn_hidden // 2, batch_first=True, bidirectional=True)
            self.node_in = gcn_hidden + 1   # + coverage scalar
        else:
            self.node_in = embed_dim + 1

        self.gcn_hidden = gcn_hidden
        self.gcn1 = DenseGCNLayer(self.node_in, gcn_hidden)
        self.gcn2 = DenseGCNLayer(gcn_hidden, gcn_hidden)

        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * gcn_hidden + edge_feat_dim, edge_hidden),
            nn.ReLU(),
            nn.Linear(edge_hidden, 1),
        )

    def encode_nodes(self, seq_tokens, x_cov):
        """
        seq_tokens: [N, K] tokenized k-mer
        x_cov     : [N, 1]
        returns   : [N, node_in]
        """
        E = self.embed(seq_tokens)  # [N, K, D]
        mask = (seq_tokens != 0).float().unsqueeze(-1)  # [N,K,1]
        if self.use_lstm:
            # packless BiLSTM (K is small)
            lengths = mask.squeeze(-1).sum(dim=1).clamp(min=1).long()
            # simple masked-mean before LSTM init (optional)
            mean_init = (E * mask).sum(dim=1) / lengths.clamp(min=1).unsqueeze(-1).float()
            H0, _ = self.lstm(E)              # [N,K, gcn_hidden]
            # take masked mean over time:
            Hseq = (H0 * mask).sum(dim=1) / lengths.clamp(min=1).unsqueeze(-1).float()  # [N, gcn_hidden]
        else:
            # masked mean over tokens
            lengths = mask.squeeze(-1).sum(dim=1).clamp(min=1)
            Hseq = (E * mask).sum(dim=1) / lengths.unsqueeze(-1)

        X = torch.cat([Hseq, x_cov], dim=1)  # [N, node_in]
        return X

    def forward(self, data):
        """
        data: PyG batch with fields:
          - seq_tokens [N,K], x_cov [N,1], edge_index [2,E], edge_attr [E,5],
            batch [N] (graph ids), num_nodes, etc.
        returns:
          logits [E]  (edge-wise)
        """
        device = data.seq_tokens.device
        N = data.num_nodes
        E = data.edge_index.size(1)

        X0 = self.encode_nodes(data.seq_tokens, data.x_cov)  # [N, node_in]

        # Per-graph GCN (dense adjacency per graph)
        out_node = X0.new_zeros((N, self.gcn_hidden))  # <- matches H width
        batch_vec = data.batch if hasattr(data, "batch") else torch.zeros(N, dtype=torch.long, device=device)
        num_graphs = int(batch_vec.max().item()) + 1 if N > 0 else 0

        # Pre-allocate a node->local index map used inside the loop
        for g in range(num_graphs):
            node_ids = (batch_vec == g).nonzero(as_tuple=False).view(-1)
            n_nodes = int(node_ids.numel())
            if n_nodes == 0:
                continue

            # map global->local node indices for this graph
            local_map = torch.full((N,), -1, dtype=torch.long, device=device)
            local_map[node_ids] = torch.arange(n_nodes, device=device, dtype=torch.long)

            # select edges fully inside this graph and remap to local ids
            ei = data.edge_index
            keep = (local_map[ei[0]] >= 0) & (local_map[ei[1]] >= 0)
            keep_idx = keep.nonzero(as_tuple=False).view(-1)
            if keep_idx.numel() == 0:
                # still run linear on isolated nodes
                H = F.relu(self.gcn1(X0[node_ids], edge_index_local=torch.empty(2,0,device=device,dtype=torch.long), n_nodes=n_nodes))
                H = F.relu(self.gcn2(H,          edge_index_local=torch.empty(2,0,device=device,dtype=torch.long), n_nodes=n_nodes))
                out_node[node_ids] = H
                continue

            src_local = local_map[ei[0, keep_idx]]
            dst_local = local_map[ei[1, keep_idx]]
            edge_local = torch.stack([src_local, dst_local], dim=0)  # [2, E_g]

            H = F.relu(self.gcn1(X0[node_ids], edge_local, n_nodes))
            H = F.relu(self.gcn2(H,            edge_local, n_nodes))
            # H width is self.gcn_hidden -> matches out_node second dim
            out_node[node_ids] = H

        # Edge logits across the whole batch
        if E == 0:
            return torch.empty((0,), device=device)

        u, v = data.edge_index
        U = out_node[u]  # [E, hidden]
        V = out_node[v]  # [E, hidden]
        EA = data.edge_attr if hasattr(data, "edge_attr") and data.edge_attr.numel() > 0 \
             else out_node.new_zeros((E, 5))
        feats = torch.cat([U, V, EA], dim=1)  # [E, 2*hidden + 5]
        logits = self.edge_mlp(feats).squeeze(-1)  # [E]
        return logits

Using DirectML: privateuseone:0


In [26]:
device = torch_directml.device() if USE_DIRECTML else (
    'cuda' if torch.cuda.is_available() else 'cpu'
)

model = HyperbubbleGNN(
    vocab_size=5, embed_dim=32, gcn_hidden=64, edge_hidden=64, edge_feat_dim=5, use_lstm=False
).to(device)

optim = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch(model, loader):
    model.train()
    tot, cnt = 0.0, 0
    for data in loader:
        data = data.to(device)
        logits = model(data)  # [E]
        if data.y_edge_mask.numel() == 0:
            continue
        loss = F.binary_cross_entropy_with_logits(
            logits, data.y_edge_mask.float(), reduction='mean'
        )
        optim.zero_grad()
        loss.backward()
        optim.step()
        tot += loss.item()
        cnt += 1
    return tot / max(cnt, 1)

@torch.no_grad()
def eval_loader(model, loader, max_batches=1):
    model.eval()
    for bi, data in enumerate(loader):
        if bi >= max_batches: break
        data = data.to(device)
        logits = model(data)
        if data.y_edge_mask.numel() == 0: continue
        preds = (logits.sigmoid() > 0.5).long()
        mask  = data.y_edge_mask.long()
        valid = int(mask.numel())
        acc = float((preds == mask).sum().item()) / max(valid, 1)
        print(f"[eval] batch {bi}  E={valid}  acc={acc:.3f}")


In [27]:
train_one_epoch(model, loader)
eval_loader(model,loader)

[eval] batch 0  E=82  acc=0.634
